# Ranking concentration through partial sums

## Ordered Partial Sums

Let $x_{[1]}\geq\cdots\geq x_{[n]}$ and $y_{[1]}\geq\cdots\geq y_{[n]}$ denote the components of two vectors after sorting from largest to smallest. We say that $y$ is majorized by $x$ when

$$
\sum_{i=1}^{k}y_{[i]}\leq\sum_{i=1}^{k}x_{[i]},\qquad k=1,\ldots,n-1,
$$

and their totals agree:

$$
\sum_{i=1}^{n}y_i=\sum_{i=1}^{n}x_i.
$$

Here, $n$ is the vector length, $k$ is a proper partial sum position, and the brackets indicate ordered components rather than time. The relation compares concentration without depending on the original component labels.

In [1]:
from io import StringIO
from pathlib import Path
from zipfile import ZipFile
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.optimization import birkhoff_von_neumann_decomposition, is_doubly_stochastic, is_majorized, majorization_matrix

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "french_49_industries")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "24_portfolio_majorization"
TABLES = BOOK_ROOT / "outputs" / "tables" / "24_portfolio_majorization"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "24_portfolio_majorization"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)


def ordered(values):
    return np.sort(np.asarray(values, dtype=float))[::-1]


def partial_sums(values):
    return np.cumsum(ordered(values))


def concentration_record(name, values):
    weights = np.asarray(values, dtype=float)
    positive = weights[weights > 0]
    squared = float(np.sum(weights ** 2))
    entropy = float(-np.sum(positive * np.log(positive)))
    return {
        "Vector": name,
        "Largest component": float(np.max(weights)),
        "Squared sum": squared,
        "Fourth power sum": float(np.sum(weights ** 4)),
        "Effective count": float(1 / squared),
        "Entropy effective count": float(np.exp(entropy)),
    }


def load_public_returns():
    expected_hash = "8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71A7C1CD20BAB83040DE"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    with ZipFile(DATA_PATH) as archive:
        member = archive.namelist()[0]
        lines = archive.read(member).decode("latin1").splitlines()
    title_index = next(index for index, line in enumerate(lines) if "Average Value Weighted Returns -- Daily" in line)
    header_index = title_index + 1
    end_index = next(index for index in range(header_index + 1, len(lines)) if not lines[index].strip())
    frame = pd.read_csv(StringIO("\n".join(lines[header_index:end_index])))
    frame = frame.rename(columns={frame.columns[0]: "Date"})
    frame["Date"] = pd.to_datetime(frame["Date"].astype(str), format="%Y%m%d")
    industries = list(frame.columns[1:17])
    frame[industries] = frame[industries].apply(pd.to_numeric, errors="coerce")
    frame[industries] = frame[industries].mask(frame[industries] <= -99)
    frame = frame.loc[(frame["Date"] >= "2019-01-01") & (frame["Date"] <= "2025-12-31"), ["Date", *industries]].dropna().reset_index(drop=True)
    assert len(industries) == 16
    assert len(frame) == 1760
    return frame, industries

vectors = {
    "Equal": np.array([0.25, 0.25, 0.25, 0.25]),
    "Intermediate": np.array([0.40, 0.30, 0.20, 0.10]),
    "Concentrated": np.array([0.55, 0.25, 0.15, 0.05]),
}

notation = pd.DataFrame([
    {"Symbol": "x", "Description": "Source vector whose concentration is the reference"},
    {"Symbol": "y", "Description": "Target vector being compared with the source"},
    {"Symbol": "x_[i]", "Description": "Component i after sorting x from largest to smallest"},
    {"Symbol": "k", "Description": "Proper partial sum position"},
    {"Symbol": "D", "Description": "Doubly stochastic matrix when y equals D times x"},
])
save_table(notation, "majorization_notation", "Notation for finite vector majorization.", "tab:chapter25-majorization-notation", fit=True)

vector_table = pd.DataFrame([
    {"Vector": name, **{f"Component {index + 1}": value for index, value in enumerate(values)}, "Total": values.sum()}
    for name, values in vectors.items()
])
save_table(vector_table, "example_vectors", "Vectors with four components used in the exact examples.", "tab:chapter25-example-vectors")

partial_table = pd.DataFrame({"Position": np.arange(1, 5), **{name: partial_sums(values) for name, values in vectors.items()}})
save_table(partial_table, "ordered_partial_sums_table", "Ordered partial sums for the exact examples.", "tab:chapter25-ordered-partial-sums")
partial_table.to_csv(FIGURE_DATA / "ordered_partial_sums.csv", index=False)

relations = []
for target_name, target in vectors.items():
    for source_name, source in vectors.items():
        relations.append({
            "Target": target_name,
            "Source": source_name,
            "Target majorized by source": is_majorized(target, source),
        })
relation_table = pd.DataFrame(relations)
save_table(relation_table, "pairwise_majorization", "Pairwise majorization results for the exact examples.", "tab:chapter25-pairwise-majorization")

figure, axis = plt.subplots(figsize=(9.4, 5.4), constrained_layout=True)
colors = [PALETTE["green"], PALETTE["amber"], PALETTE["rose"]]
for (name, values), color in zip(vectors.items(), colors):
    axis.plot(np.arange(0, 5), np.r_[0, partial_sums(values)], marker="o", linewidth=2, color=color, label=name)
axis.set_xlabel("Number of largest components")
axis.set_ylabel("Ordered partial sum")
axis.set_title("Ordered Partial Sums Reveal Concentration")
axis.set_xticks(np.arange(0, 5))
axis.legend()
finish_axis(axis)
save_figure(figure, "ordered_partial_sums")
plt.close(figure)

gaps = pd.DataFrame({
    "Position": np.arange(1, 5),
    "Equal minus concentrated": partial_sums(vectors["Equal"]) - partial_sums(vectors["Concentrated"]),
    "Intermediate minus concentrated": partial_sums(vectors["Intermediate"]) - partial_sums(vectors["Concentrated"]),
})
gaps.to_csv(FIGURE_DATA / "partial_sum_gaps.csv", index=False)
figure, axis = plt.subplots(figsize=(9.4, 5.2), constrained_layout=True)
axis.plot(gaps["Position"], gaps["Equal minus concentrated"], marker="o", color=PALETTE["cyan"], linewidth=2, label="Equal target")
axis.plot(gaps["Position"], gaps["Intermediate minus concentrated"], marker="s", color=PALETTE["amber"], linewidth=2, label="Intermediate target")
axis.axhline(0, color=PALETTE["text"], linewidth=1)
axis.set_xlabel("Partial sum position")
axis.set_ylabel("Target sum minus source sum")
axis.set_title("Majorized Targets Do Not Cross Above the Source")
axis.set_xticks(np.arange(1, 5))
axis.legend()
finish_axis(axis)
save_figure(figure, "partial_sum_gaps")
plt.close(figure)

,Symbol,Description
0,x,Source vector whose concentration is the refer...
1,y,Target vector being compared with the source
2,x_[i],Component i after sorting x from largest to sm...
3,k,Proper partial sum position
4,D,Doubly stochastic matrix when y equals D times x


,Vector,Component 1,Component 2,Component 3,Component 4,Total
0,Equal,0.25,0.25,0.25,0.25,1
1,Intermediate,0.4,0.3,0.2,0.1,1
2,Concentrated,0.55,0.25,0.15,0.05,1


,Position,Equal,Intermediate,Concentrated
0,1,0.25,0.4,0.55
1,2,0.5,0.7,0.8
2,3,0.75,0.9,0.95
3,4,1,1,1


,Target,Source,Target majorized by source
0,Equal,Equal,Pass
1,Equal,Intermediate,Pass
2,Equal,Concentrated,Pass
3,Intermediate,Equal,Fail
4,Intermediate,Intermediate,Pass
5,Intermediate,Concentrated,Pass
6,Concentrated,Equal,Fail
7,Concentrated,Intermediate,Fail
8,Concentrated,Concentrated,Pass


<Figure size 940x540 with 1 Axes>

<Figure size 940x520 with 1 Axes>

## Convex Measures of Concentration

For a nonnegative portfolio weight vector $w$ with total one, the squared concentration and its reciprocal are

$$
H(w)=\sum_{i=1}^{n}w_i^2,
\qquad
N_{\mathrm{eff}}(w)=\frac{1}{H(w)}.
$$

Here, $w_i$ is the weight of asset $i$, $H(w)$ is the squared concentration, and $N_{\mathrm{eff}}(w)$ is the effective number of equally weighted positions. A larger $H(w)$ and a smaller $N_{\mathrm{eff}}(w)$ indicate greater concentration. These summaries do not record asset dependence or expected returns.

In [2]:
concentration = pd.DataFrame([concentration_record(name, values) for name, values in vectors.items()])
save_table(concentration, "concentration_statistics", "Convex concentration summaries for the exact examples.", "tab:chapter25-concentration-statistics", fit=True)
concentration.to_csv(FIGURE_DATA / "concentration_measures.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.9), constrained_layout=True)
axes[0].bar(concentration["Vector"], concentration["Squared sum"], color=[PALETTE["green"], PALETTE["amber"], PALETTE["rose"]])
axes[0].set_ylabel("Squared concentration")
axes[0].set_title("Concentration")
axes[1].bar(concentration["Vector"], concentration["Effective count"], color=[PALETTE["green"], PALETTE["amber"], PALETTE["rose"]])
axes[1].set_ylabel("Effective count")
axes[1].set_title("Reciprocal Concentration")
for axis in axes:
    axis.tick_params(axis="x", rotation=18)
    finish_axis(axis)
save_figure(figure, "concentration_measures")
plt.close(figure)

incomparable_a = np.array([0.50, 0.20, 0.20, 0.10])
incomparable_b = np.array([0.40, 0.40, 0.10, 0.10])
incomparable = pd.DataFrame({
    "Position": np.arange(1, 5),
    "Vector A": partial_sums(incomparable_a),
    "Vector B": partial_sums(incomparable_b),
    "A minus B": partial_sums(incomparable_a) - partial_sums(incomparable_b),
})
save_table(incomparable, "incomparable_pair", "Crossing ordered partial sums for an incomparable pair.", "tab:chapter25-incomparable-pair")
incomparable.to_csv(FIGURE_DATA / "incomparable_partial_sums.csv", index=False)

figure, axis = plt.subplots(figsize=(9.2, 5.2), constrained_layout=True)
axis.plot(np.arange(0, 5), np.r_[0, partial_sums(incomparable_a)], marker="o", color=PALETTE["cyan"], linewidth=2, label="Vector A")
axis.plot(np.arange(0, 5), np.r_[0, partial_sums(incomparable_b)], marker="s", color=PALETTE["violet"], linewidth=2, label="Vector B")
axis.set_xlabel("Number of largest components")
axis.set_ylabel("Ordered partial sum")
axis.set_title("Crossing Partial Sums Give No Majorization Order")
axis.set_xticks(np.arange(0, 5))
axis.legend()
finish_axis(axis)
save_figure(figure, "incomparable_partial_sums")
plt.close(figure)

,Vector,Largest component,Squared sum,Fourth power sum,Effective count,Entropy effective count
0,Equal,0.25,0.25,0.015625,4,4
1,Intermediate,0.4,0.3,0.0354,3.333333,3.596115
2,Concentrated,0.55,0.39,0.095925,2.564103,3.033565


<Figure size 1080x490 with 2 Axes>

,Position,Vector A,Vector B,A minus B
0,1,0.5,0.4,0.1
1,2,0.7,0.8,-0.1
2,3,0.9,0.9,-1.11022e-16
3,4,1,1,-1.11022e-16


<Figure size 920x520 with 1 Axes>

## Comparing signed-vector permutations

Majorization is defined for real vectors with equal totals, so components need not be positive. Sorting also makes the relation invariant to a relabeling of components. For the signed target $y=(-1,0,1)$ and source $x=(-2,0,2)$, the proper ordered partial sums of $y$ remain below those of $x$, while both totals equal zero.

In [3]:
signed_target = np.array([-1.0, 0.0, 1.0])
signed_source = np.array([-2.0, 0.0, 2.0])
signed = pd.DataFrame({
    "Position": np.arange(1, 4),
    "Target ordered component": ordered(signed_target),
    "Source ordered component": ordered(signed_source),
    "Target partial sum": partial_sums(signed_target),
    "Source partial sum": partial_sums(signed_source),
})
signed["Target majorized by source"] = is_majorized(signed_target, signed_source)
save_table(signed, "signed_vector_checks", "Ordered components and partial sums for a signed example.", "tab:chapter25-signed-vector-checks", fit=True)

permutations = [np.array([0, 1, 2, 3]), np.array([3, 2, 1, 0]), np.array([1, 3, 0, 2])]
permutation_rows = []
for order_index, order_indexing in enumerate(permutations, start=1):
    permuted = vectors["Intermediate"][order_indexing]
    permutation_rows.append({
        "Relabeling": order_index,
        "Order": " ".join(str(value + 1) for value in order_indexing),
        "Majorized by concentrated": is_majorized(permuted, vectors["Concentrated"]),
        "Squared sum": np.sum(permuted ** 2),
        "Largest component": np.max(permuted),
    })
permutation_table = pd.DataFrame(permutation_rows)
save_table(permutation_table, "permutation_invariance_checks", "Majorization and concentration after three component relabelings.", "tab:chapter25-permutation-invariance")

assert is_majorized(vectors["Equal"], vectors["Intermediate"])
assert is_majorized(vectors["Intermediate"], vectors["Concentrated"])
assert not is_majorized(incomparable_a, incomparable_b)
assert not is_majorized(incomparable_b, incomparable_a)
assert is_majorized(signed_target, signed_source)
assert concentration.set_index("Vector").loc["Equal", "Squared sum"] < concentration.set_index("Vector").loc["Concentrated", "Squared sum"]
display(permutation_table)

,Position,Target ordered component,Source ordered component,Target partial sum,Source partial sum,Target majorized by source
0,1,1,2,1,2,Pass
1,2,0,0,1,2,Pass
2,3,-1,-2,0,0,Pass


,Relabeling,Order,Majorized by concentrated,Squared sum,Largest component
0,1,1 2 3 4,Pass,0.3,0.4
1,2,4 3 2 1,Pass,0.3,0.4
2,3,2 4 1 3,Pass,0.3,0.4


,Relabeling,Order,Majorized by concentrated,Squared sum,Largest component
0,1,1 2 3 4,True,0.3,0.4
1,2,4 3 2 1,True,0.3,0.4
2,3,2 4 1 3,True,0.3,0.4
